# check

> Check a learner's function with small tests. Validate test text, run it with time limits, and share results without test source or expected values.

In [1]:
#| default_exp check

In [2]:
#| export
import ast, builtins, ctypes, inspect, io, json, os, signal, subprocess, sys, threading, time
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from fastcore.basics import AttrDict, adict, stop
from fastcore.nbio import read_nb, dict2nb

In [3]:
#| hide
from fastcore.test import test_eq, test_fail, expect_fail
from fastcore.nbio import mk_cell, new_nb

## Safe tests

The model writes the tests, so the notebook does not trust them. `safe_test` accepts only one `assert`, a statement that checks a condition.
The assert can use the learner's function, literals (fixed values such as `2`), comparisons, arithmetic and a short list of builtins (Python's built-in functions).
It cannot use an import, a private attribute (a name that starts with `_`), `open`, `exec` or `eval`.
`validate_exercise` checks the shape of the model's reply. Tests run with only the safe builtins. This check limits the test text; it does not isolate the learner's function from the rest of Python.

In [4]:
#| export
SAFE_NAMES = ('abs', 'all', 'any', 'bool', 'dict', 'enumerate', 'float', 'int', 'isinstance', 'len', 'list', 'max', 'min',
              'range', 'reversed', 'round', 'set', 'sorted', 'str', 'sum', 'tuple', 'zip', 'True', 'False', 'None')
SAFE_BUILTINS = {n: getattr(builtins, n) for n in SAFE_NAMES if hasattr(builtins, n)}
SAFE_NODES = (ast.Module, ast.Assert, ast.Expr, ast.Compare, ast.Call, ast.Name, ast.Load, ast.Constant, ast.List, ast.Tuple, ast.Dict,
              ast.Set, ast.BinOp, ast.UnaryOp, ast.BoolOp, ast.Subscript, ast.Slice, ast.keyword, ast.Attribute, ast.IfExp,
              ast.operator, ast.unaryop, ast.boolop, ast.cmpop, ast.ListComp, ast.GeneratorExp, ast.comprehension, ast.Store)
TEST_SECONDS = 2.

def safe_test(t:str, fname:str) -> bool:
    "Whether `t` is one plain `assert` that uses only `fname`, literals and the safe builtins."
    try: tree = ast.parse(t)
    except SyntaxError: return False
    if len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assert): return False
    bound = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}   # names of a comprehension
    for n in ast.walk(tree):
        if not isinstance(n, SAFE_NODES): return False
        if isinstance(n, ast.Name) and n.id not in (fname, *SAFE_NAMES, *bound): return False
        if isinstance(n, ast.Attribute) and n.attr.startswith('_'): return False
        if isinstance(n, ast.Constant) and isinstance(n.value, str) and len(n.value) > 2000: return False
    return True

In [5]:
#| export
def validate_exercise(r) -> AttrDict:
    "Check the model's exercise and raise `ValueError` with the reason if invalid."
    if not isinstance(r, dict): raise ValueError('the reply is not a JSON object')
    task, fname, stub, tests = r.get('task'), r.get('fname'), r.get('stub'), r.get('tests')
    if not (isinstance(task, str) and task.strip()): raise ValueError('`task` must be a text')
    if not (isinstance(fname, str) and fname.isidentifier() and not fname.startswith('_')): raise ValueError('`fname` must be a plain function name')
    if not isinstance(stub, str): raise ValueError('`stub` must be a text')
    if not (isinstance(tests, list) and 1 <= len(tests) <= 12): raise ValueError('`tests` must be a list of 1 to 12 texts or test objects')
    for t in tests:
        if isinstance(t, dict):
            if not isinstance(t.get('say', ''), str) or not isinstance(t.get('hidden', False), bool):
                raise ValueError('a test object needs text `say` and boolean `hidden`')
            t = t.get('test')
        if not isinstance(t, str): raise ValueError('`tests` must be a list of 1 to 12 texts or test objects')
        if not safe_test(t, fname): raise ValueError(f'a test is not one plain assert on `{fname}`: {t[:120]}')
    return AttrDict(task=task, fname=fname, stub=stub, tests=tests, solution=r.get('solution', '') if isinstance(r.get('solution'), str) else '')

`safe_test` refuses an import, a name with two leading underscores, a call to anything but the function and the safe builtins, and code that is not one assert.

In [6]:
assert safe_test("assert rrf([['a'], ['b']], k=0) == ['a', 'b']", 'rrf')
assert safe_test("assert all(x in 'ab' for x in rrf([['a', 'b']]))", 'rrf')
for bad in ("import os", "assert __import__('os')", "assert rrf.__globals__", "assert open('/etc/passwd')", "assert exec('1')",
            "rrf([])", "while True: pass", "assert rrf(1); import os", "assert (lambda: 1)()"):
    assert not safe_test(bad, 'rrf'), bad
safe_test('assert rrf([]) == []', 'rrf')

True

`validate_exercise` refuses an empty or a text `tests`, a missing name, and an unsafe test, with a reason. A test object is a dictionary with `test` text, an optional `say` label, and an optional `hidden` flag.

In [7]:
good = dict(task='t', fname='f', stub='def f(): ...', tests=['assert f() == 1'])
test_eq(validate_exercise(good).fname, 'f')
for bad, why in ((dict(good, tests=[]), 'list of 1 to 12'), (dict(good, tests='assert f()'), 'list of 1 to 12'),
                 ({k: v for k, v in good.items() if k != 'fname'}, 'fname'), (dict(good, tests=['import os']), 'not one plain assert'), ([], 'JSON object')):
    test_fail(lambda: validate_exercise(bad), contains=why)
validate_exercise(good)

```python
{ 'fname': 'f',
  'solution': '',
  'stub': 'def f(): ...',
  'task': 't',
  'tests': ['assert f() == 1']}
```

Named tests use the same safety check as text tests. Missing labels and hidden flags default to an empty label and `False`.

In [8]:
named = dict(good, tests=[dict(test='assert f() == 1', say='One', hidden=True)])
test_eq(validate_exercise(named).tests, named['tests'])
validate_exercise(named).tests

[{'test': 'assert f() == 1', 'say': 'One', 'hidden': True}]

## Time limits

`time_limit` raises `TimeoutError` when the time is up. On the main thread it uses `SIGALRM`, an operating-system alarm. On another thread it starts a watchdog, a timer that asks Python to raise the error in the test thread. The watchdog cannot interrupt a call blocked in C code. It takes effect when that call returns to Python. The timer is always cancelled when the context ends.

In [9]:
#| export
@contextmanager
def time_limit(secs:float=TEST_SECONDS):
    "Raise `TimeoutError` after `secs`; worker threads cannot interrupt blocked C calls."
    if secs <= 0: raise TimeoutError('the test has no time left')
    if threading.current_thread() is threading.main_thread():
        def boom(*_): raise TimeoutError(f'the test ran longer than {secs:g} s')
        old = signal.signal(signal.SIGALRM, boom)
        try:
            signal.setitimer(signal.ITIMER_REAL, secs)
            yield
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old)
        return
    ident, lock, active = threading.get_ident(), threading.Lock(), True
    def expire():
        with lock:
            if not active: return
            changed = ctypes.pythonapi.PyThreadState_SetAsyncExc(ctypes.c_ulong(ident), ctypes.py_object(TimeoutError))
            if changed > 1:
                ctypes.pythonapi.PyThreadState_SetAsyncExc(ctypes.c_ulong(ident), ctypes.c_void_p())
    timer = threading.Timer(secs, expire)
    timer.daemon = True
    try:
        timer.start()
        yield
    finally:
        with lock:
            active = False
            timer.cancel()
        timer.join()

A test that runs too long fails, and Python continues. The loop below has its own one-second limit, so even a broken timeout cannot hang this example.

In [10]:
def bounded_loop():
    deadline, steps = time.monotonic() + 1., 0
    while time.monotonic() < deadline:
        steps += 1

started = time.monotonic()
with expect_fail(TimeoutError):
    with time_limit(.2): bounded_loop()
elapsed = time.monotonic() - started
assert elapsed < 1.
round(elapsed, 2)

0.21

## Run and share results

`Report` is an `AttrDict`, a dictionary whose keys can also be read as attributes. `found` says whether the function is callable. `timeout` says whether any check ran out of time or the run was interrupted. `checks` holds each result; `passed` is true only when a found function passes every check. Each check has a zero-based index `i`, a label, a status, `got`, error text, a `hidden` flag and elapsed milliseconds `ms`. The status is `pass`, `fail` (a false assert), `error` (another exception) or `timeout`.

In [11]:
#| export
class Report(AttrDict):
    "Results for one learner function."
    @property
    def passed(self):
        return self.found and bool(self.checks) and all(c.status == 'pass' for c in self.checks)

def public(report) -> dict:
    "Return page-safe results without test source or expected values."
    checks = []
    for c in report.checks:
        error = c.error.split(':', 1)[0] if c.hidden else c.error
        checks.append(dict(i=c.i, label=c.label, status=c.status, got=c.got, error=error, hidden=c.hidden, ms=c.ms))
    return dict(v=1, found=report.found, timeout=report.timeout, checks=checks)

`run_asserts` gives each test `secs` seconds within a shared `total` budget. Tests left after that budget expires become timeouts without running. `KeyboardInterrupt` stops the run and marks the current and remaining checks as timeouts. Printed output and error output are captured and discarded. For a single equality assert with a call on the left, the call runs once and its value becomes `got`, clipped to 200 characters.

In [12]:
#| export
def run_asserts(fn, fname, tests, *, secs=2., total=10.) -> Report:
    "Run tests with per-test and total time limits, capturing output and actual values."
    report = Report(found=callable(fn), timeout=False, checks=[])
    deadline, stopped = time.monotonic() + total, False
    for i, spec in enumerate(tests):
        t = spec if isinstance(spec, str) else spec['test']
        hidden = False if isinstance(spec, str) else spec.get('hidden', False)
        label = '' if isinstance(spec, str) else spec.get('say', '')
        check = AttrDict(i=i, label=label or f'Test {i + 1}', status='pass', got=None, error='', hidden=hidden, ms=0.)
        started = time.monotonic()
        remaining = deadline - started
        if stopped or remaining <= 0:
            check.status, check.error = 'timeout', 'TimeoutError' if hidden else 'TimeoutError: the run stopped or spent its budget'
        else:
            env = {'__builtins__': SAFE_BUILTINS, fname: fn}
            try:
                with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()), time_limit(min(secs, remaining)):
                    if not safe_test(t, fname): raise ValueError('the test is not one safe assert')
                    tree = ast.parse(t)
                    condition = tree.body[0].test
                    capture = (isinstance(condition, ast.Compare) and len(condition.ops) == 1
                               and isinstance(condition.ops[0], ast.Eq) and isinstance(condition.left, ast.Call))
                    if capture:
                        condition.left = ast.NamedExpr(target=ast.Name(id='_abhyasa_actual', ctx=ast.Store()), value=condition.left)
                        ast.fix_missing_locations(tree)
                    try: exec(compile(tree, '<learner test>', 'exec'), env)
                    except (TimeoutError, KeyboardInterrupt): raise
                    except Exception as e:
                        check.status = 'fail' if isinstance(e, AssertionError) else 'error'
                        check.error = type(e).__name__ if hidden else f'{type(e).__name__}: {e}'[:200]
                    if capture and '_abhyasa_actual' in env: check.got = repr(env['_abhyasa_actual'])[:200]
            except KeyboardInterrupt:
                stopped = True
                check.status, check.error = 'timeout', 'KeyboardInterrupt' if hidden else 'KeyboardInterrupt: the run stopped'
            except TimeoutError:
                check.status, check.error = 'timeout', 'TimeoutError' if hidden else 'TimeoutError: the test ran out of time'
            except Exception as e:
                check.status = 'error'
                check.error = type(e).__name__ if hidden else f'{type(e).__name__}: {e}'[:200]
        check.ms = (time.monotonic() - started) * 1000
        report.timeout |= check.status == 'timeout'
        report.checks.append(check)
    return report

A report holds one result for each test. `got` is the value returned by the call, written as Python text. `say` gives a short label.

In [13]:
def double(n): return n * 2

result = run_asserts(double, 'double', ['assert double(2) == 4', dict(test='assert double(3) == 7', say='Three')])
test_eq([c.status for c in result.checks], ['pass', 'fail'])
test_eq([c.got for c in result.checks], ['4', '6'])
assert not result.passed
result

```python
{ 'checks': [ { 'error': '',
                'got': '4',
                'hidden': False,
                'i': 0,
                'label': 'Test 1',
                'ms': 0.25216699577867985,
                'status': 'pass'},
              { 'error': 'AssertionError: ',
                'got': '6',
                'hidden': False,
                'i': 1,
                'label': 'Three',
                'ms': 0.16520917415618896,
                'status': 'fail'}],
  'found': True,
  'timeout': False}
```

A hidden test keeps its expected value private. `public` makes a page-safe dictionary. It omits test text and keeps only the exception class for a hidden error.

In [14]:
private_test = 'assert double(3) == 912839'
private_result = run_asserts(double, 'double', [dict(test=private_test, say='Private case', hidden=True)])
page = public(private_result)
assert private_test not in json.dumps(page) and '912839' not in json.dumps(page)
test_eq(page['checks'][0]['error'], 'AssertionError')
page

{'v': 1,
 'found': True,
 'timeout': False,
 'checks': [{'i': 0,
   'label': 'Private case',
   'status': 'fail',
   'got': '6',
   'error': 'AssertionError',
   'hidden': True,
   'ms': 0.14683394692838192}]}

## Notebook test cells

`load_tests` reads a notebook path or an in-memory notebook. A code cell with `#| setup` supplies a fixture. A cell with `#| test: Label` supplies a check; a bare `#| test` gets a numbered label. The `abhyasa` metadata can also mark tests, labels, and hidden checks. Comment directives take priority.

These cells are trusted author code, with full Python builtins. Do not pass model output to this runner.

In [15]:
#| export
def load_tests(nb) -> tuple:
    "Return ordered setup sources and labelled test cells from a notebook."
    nb = read_nb(nb) if isinstance(nb, (str, os.PathLike)) else dict2nb(nb)
    setup, tests = [], []
    for cell in nb.cells:
        if cell.cell_type != 'code': continue
        directives = cell.directives
        meta = cell.get('metadata', {}).get('abhyasa', {})
        if 'setup' in directives: setup.append(cell.source)
        if 'test' not in directives and not meta.get('test', False): continue
        label = directives['test'] if 'test' in directives else meta.get('say', '')
        tests.append(dict(source=cell.source, say=label or f'Test {len(tests) + 1}',
                          hidden=True if 'hidden' in directives else meta.get('hidden', False)))
    return setup, tests

`run_cells` runs setup once in a copy of the supplied namespace. Each check gets a fresh shallow copy of that base: new names do not pass between tests, but shared mutable objects are not copied. Setup and tests share one total time budget. Setup exceptions mark all checks with `SetupError`; an interrupt stops the run.

A failing top-level `assert call(...) == expected` records the actual value without calling the function twice. Hidden failures show only the exception class, not its message or the test source.

In [16]:
#| export
def run_cells(tests, ns, *, setup=(), secs=2., total=10.) -> Report:
    "Run trusted test cells in separate shallow copies of one setup namespace."
    report = Report(found=True, timeout=False, checks=[])
    deadline, stopped, setup_error = time.monotonic() + total, False, ''
    base = dict(ns)
    base['__builtins__'] = builtins.__dict__
    try:
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            for source in setup:
                with time_limit(min(secs, deadline - time.monotonic())): exec(source, base)
    except KeyboardInterrupt: stopped = True
    except TimeoutError: stopped = True
    except (Exception, SystemExit) as e: setup_error = f'SetupError: {type(e).__name__}'
    for i, spec in enumerate(tests):
        hidden = spec.get('hidden', False)
        check = AttrDict(i=i, label=spec.get('say') or f'Test {i + 1}', status='pass', got=None,
                         error='', hidden=hidden, ms=0.)
        started = time.monotonic()
        remaining = deadline - started
        if setup_error:
            check.status, check.error = 'error', 'SetupError' if hidden else setup_error
        elif stopped or remaining <= 0:
            check.status, check.error = 'timeout', 'TimeoutError' if hidden else 'TimeoutError: the run stopped or spent its budget'
        else:
            env = dict(base)
            try:
                with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()), time_limit(min(secs, remaining)):
                    tree = ast.parse(spec['source'])
                    name = '_abhyasa_actual'
                    used = set(env) | {n.id for n in ast.walk(tree) if isinstance(n, ast.Name)}
                    captures = {}
                    for statement in tree.body:
                        if not isinstance(statement, ast.Assert): continue
                        condition = statement.test
                        if (isinstance(condition, ast.Compare) and len(condition.ops) == 1
                            and isinstance(condition.ops[0], ast.Eq) and isinstance(condition.left, ast.Call)):
                            while name in used: name += '_'
                            used.add(name)
                            captures[(statement.lineno, statement.col_offset), (statement.end_lineno, statement.end_col_offset)] = name
                            condition.left = ast.NamedExpr(target=ast.Name(id=name, ctx=ast.Store()), value=condition.left)
                    code = compile(ast.fix_missing_locations(tree), '<notebook test>', 'exec')
                    try: exec(code, env)
                    except AssertionError as e:
                        check.status = 'fail'
                        check.error = type(e).__name__ if hidden else f'{type(e).__name__}: {e}'[:200]
                        tb = e.__traceback__
                        while tb and tb.tb_frame.f_code is not code: tb = tb.tb_next
                        position = list(code.co_positions())[tb.tb_lasti // 2] if tb else (None, None, None, None)
                        captured = next((name for (start, end), name in captures.items()
                                         if position[0] is not None and start <= (position[0], position[2]) < end), None)
                        if captured in env: check.got = repr(env[captured])[:200]
            except KeyboardInterrupt:
                stopped = True
                check.status, check.error = 'timeout', 'KeyboardInterrupt' if hidden else 'KeyboardInterrupt: the run stopped'
            except TimeoutError:
                check.status, check.error = 'timeout', 'TimeoutError' if hidden else 'TimeoutError: the test ran out of time'
            except (Exception, SystemExit) as e:
                check.status = 'error'
                check.error = type(e).__name__ if hidden else f'{type(e).__name__}: {e}'[:200]
        check.ms = (time.monotonic() - started) * 1000
        report.timeout |= check.status == 'timeout'
        report.checks.append(check)
    report.timeout |= stopped
    return report

## Checks in a user kernel

`snippet` builds standalone Python from the actual runner functions with `inspect.getsource`. It embeds test data as JSON, needs no installed `abhyasa` or `fastcore`, and prints one JSON result line. Its definitions live in a private namespace, so no harness names remain in the user kernel. The kernel's existing names, including `__builtins__`, stay unchanged.

In [17]:
#| export
def snippet(kind, fname, tests, *, secs=2., total=10., setup=()) -> str:
    "Build a standalone kernel check that prints one public JSON result."
    if kind not in ('assert', 'cell'): raise ValueError("kind must be 'assert' or 'cell'")
    dependencies = [stop, adict, AttrDict, time_limit, public]
    runners = [safe_test, run_asserts] if kind == 'assert' else [run_cells]
    data = json.dumps(dict(fname=fname, tests=tests, secs=secs, total=total, setup=list(setup), safe_names=SAFE_NAMES))
    parts = [
        'import ast, builtins, ctypes, io, json, signal, threading, time',
        'from contextlib import contextmanager, redirect_stdout, redirect_stderr',
        f'data = json.loads({data!r})',
        'SAFE_NAMES = tuple(data["safe_names"])',
        'SAFE_BUILTINS = {n: getattr(builtins, n) for n in SAFE_NAMES if hasattr(builtins, n)}',
        'SAFE_NODES = (' + ', '.join(f'ast.{node.__name__}' for node in SAFE_NODES) + ',)',
        f'TEST_SECONDS = {TEST_SECONDS!r}',
        *(inspect.getsource(obj) for obj in dependencies),
        'class Report(AttrDict):\n' + inspect.getsource(Report.passed.fget),
        *(inspect.getsource(obj) for obj in runners),
    ]
    if kind == 'assert':
        parts.append('result = run_asserts(_ns.get(data["fname"]), data["fname"], data["tests"], secs=data["secs"], total=data["total"])')
    else:
        parts.append('result = run_cells(data["tests"], _ns, setup=data["setup"], secs=data["secs"], total=data["total"])')
    parts.append('print(json.dumps(public(result)))')
    return f'exec({chr(10).join(parts)!r}, {{"_ns": globals()}})'


`parse` finds the last JSON result line in kernel stdout and rejects text that is not a public check result. `run_isolated` runs a reference solution and the same snippet in a fresh isolated Python process. The process has an outer `total + 2` second limit, even when the solution itself never finishes. This is process isolation, not a security sandbox.

In [18]:
#| export
def parse(text) -> dict:
    "Read the last public check JSON line, or return a missing-result error."
    for line in reversed(text.splitlines()):
        try: result = json.loads(line)
        except (ValueError, TypeError): continue
        if (isinstance(result, dict) and result.get('v') == 1
            and isinstance(result.get('found'), bool) and isinstance(result.get('timeout'), bool)
            and isinstance(result.get('checks'), list)): return result
        return {'error': 'no check result'}
    return {'error': 'no check result'}

def run_isolated(kind, fname, source, tests, *, total=10., setup=()) -> dict:
    "Run a reference solution and checks in an isolated Python subprocess."
    code = source + '\n' + snippet(kind, fname, tests, total=total, setup=setup)
    try:
        result = subprocess.run([sys.executable, '-I'], input=code, capture_output=True, text=True, timeout=total + 2)
    except subprocess.TimeoutExpired: return {'timeout': True}
    return parse(result.stdout)

## Small examples

A fixture and three checks keep their notebook order. Comments supply the first label; metadata makes the second check hidden; a bare directive gets a numbered label.

In [19]:
example_nb = new_nb([
    mk_cell('#| setup\nnumber = 3'),
    mk_cell('#| test: Double three\nassert double(number) == 6'),
    mk_cell('assert double(number) == 918273', metadata={'abhyasa': {'test': True, 'say': 'Private', 'hidden': True}}),
    mk_cell('#| test\nassert number == 3'),
])
example_setup, example_tests = load_tests(example_nb)
assert len(example_setup) == 1
assert [t['say'] for t in example_tests] == ['Double three', 'Private', 'Test 3']
assert [t['hidden'] for t in example_tests] == [False, True, False]
[t['say'] for t in example_tests]

['Double three', 'Private', 'Test 3']

The fixture is available in every check. The hidden failure shares its status and actual value, but not the private expected value.

In [20]:
cell_report = run_cells(example_tests, {'double': double}, setup=example_setup)
assert [c.status for c in cell_report.checks] == ['pass', 'fail', 'pass']
assert cell_report.checks[1].got == '6'
assert '918273' not in json.dumps(public(cell_report))
[(c.label, c.status, c.error) for c in cell_report.checks]

[('Double three', 'pass', ''),
 ('Private', 'fail', 'AssertionError'),
 ('Test 3', 'pass', '')]

The same cell checks run in a kernel-like dictionary. The snippet prints one JSON line and leaves that dictionary unchanged. `parse` also gives a clear result when stdout contains no check report.

In [21]:
kernel_ns = {'double': double, '__builtins__': builtins.__dict__}
kernel_before = dict(kernel_ns)
kernel_stdout = io.StringIO()
with redirect_stdout(kernel_stdout):
    exec(snippet('cell', 'double', example_tests, setup=example_setup), kernel_ns)
kernel_result = parse(kernel_stdout.getvalue())
assert kernel_ns == kernel_before
assert len(kernel_stdout.getvalue().splitlines()) == 1
assert [c['status'] for c in kernel_result['checks']] == ['pass', 'fail', 'pass']
assert parse('not JSON') == {'error': 'no check result'}
[(c['label'], c['status']) for c in kernel_result['checks']]

[('Double three', 'pass'), ('Private', 'fail'), ('Test 3', 'pass')]